In [ ]:
# ============================================================
# 0. 라이브러리 & 경로 설정
# ============================================================
import numpy as np
import pandas as pd
from pathlib import Path
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

DATA_DIR = Path("data")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {"kpx_group_1": 21600, "kpx_group_2": 21600, "kpx_group_3": 21000}
ID_COLS = ["forecast_kst_dtm", "data_available_kst_dtm", "grid_id", "latitude", "longitude"]
CORR_VAL_START = pd.Timestamp("2024-01-01")

TRAIN_PERIOD_START = {
    "kpx_group_1": "2023-01-01",
    "kpx_group_2": "2022-01-01",
    "kpx_group_3": "2023-01-01",
}

GROUP_LDAPS_GRIDS = {
    "kpx_group_1": [5, 6, 10, 11],
    "kpx_group_2": list(range(1, 17)),
    "kpx_group_3": list(range(1, 17)),
}
GFS_GRID = 5

GROUP_SCADA_TURBINES = {
    "kpx_group_1": ("vestas", list(range(1, 7))),
    "kpx_group_2": ("vestas", list(range(7, 13))),
    "kpx_group_3": ("unison", list(range(1, 6))),
}

TOP2_MODELS = {
    "kpx_group_1": ["LightGBM", "HistGB"],
    "kpx_group_2": ["HistGB", "LightGBM"],
    "kpx_group_3": ["MLP", "RandomForest"],
}


# ============================================================
# 1. 원본 파일 로더
# ============================================================
def load_csv(path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    return df

def load_train_labels():
    df = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
    df["kst_dtm"] = pd.to_datetime(df["kst_dtm"])
    return df

def load_scada(maker):
    df = pd.read_csv(TRAIN_DIR / f"scada_{maker}_train.csv", encoding="utf-8-sig")
    df["kst_dtm"] = pd.to_datetime(df["kst_dtm"])
    return df


# ============================================================
# 2. 격자 집계 + 캘린더 + 데이터셋 조립
# ============================================================
def aggregate_grids(df, grids, prefix):
    sub = df[df["grid_id"].isin(grids)]
    value_cols = [c for c in sub.columns if c not in ID_COLS]
    agg = sub.groupby("forecast_kst_dtm")[value_cols].mean()
    agg.columns = [f"{prefix}_{c}" for c in agg.columns]
    return agg.reset_index()

def calendar_features(dt_series):
    dt = pd.to_datetime(dt_series)
    out = pd.DataFrame(index=dt.index)
    out["hour_sin"] = np.sin(2 * np.pi * dt.dt.hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * dt.dt.hour / 24)
    out["month_sin"] = np.sin(2 * np.pi * dt.dt.month / 12)
    out["month_cos"] = np.cos(2 * np.pi * dt.dt.month / 12)
    return out

def build_dataset(split, group, ldaps, gfs, labels=None):
    ldaps_agg = aggregate_grids(ldaps, GROUP_LDAPS_GRIDS[group], "ldaps")
    gfs_agg = aggregate_grids(gfs, [GFS_GRID], "gfs")

    df = ldaps_agg.merge(gfs_agg, on="forecast_kst_dtm", how="inner")
    df = pd.concat([df, calendar_features(df["forecast_kst_dtm"])], axis=1)

    if labels is not None:
        y = labels[["kst_dtm", group]].rename(columns={"kst_dtm": "forecast_kst_dtm", group: "target"})
        df = df.merge(y, on="forecast_kst_dtm", how="left").dropna(subset=["target"])

    df = df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    feat_cols = [c for c in df.columns if c not in ("forecast_kst_dtm", "target")]
    df[feat_cols] = df[feat_cols].interpolate(method="linear", limit_direction="both")
    return df


# ============================================================
# 3. 실행 - train/test 데이터셋 생성
# ============================================================
ldaps_train_raw = load_csv(TRAIN_DIR / "ldaps_train.csv")
gfs_train_raw = load_csv(TRAIN_DIR / "gfs_train.csv")
ldaps_test_raw = load_csv(TEST_DIR / "ldaps_test.csv")
gfs_test_raw = load_csv(TEST_DIR / "gfs_test.csv")
train_labels = load_train_labels()

train_datasets = {}
test_datasets = {}
for g in TARGET_COLS:
    train_datasets[g] = build_dataset("train", g, ldaps_train_raw, gfs_train_raw, train_labels)
    test_datasets[g] = build_dataset("test", g, ldaps_test_raw, gfs_test_raw, None)

RAW_FEATS = [c for c in train_datasets["kpx_group_1"].columns if c not in ("forecast_kst_dtm", "target")]

for g in TARGET_COLS:
    print(f"[{g}] train={train_datasets[g].shape}, test={test_datasets[g].shape}")
print(f"전체 피처 수: {len(RAW_FEATS)}")


# ============================================================
# 4. SCADA 로드 + 풍속 보정모델(v4) 학습
# ============================================================
scada_vestas = load_scada("vestas")
scada_unison = load_scada("unison")

def build_scada_wind_target_avg(group):
    maker, turbines = GROUP_SCADA_TURBINES[group]
    src = scada_vestas if maker == "vestas" else scada_unison
    ws_cols = [f"{maker}_wtg{t:02d}_ws" for t in turbines]

    turbine_mean = src[ws_cols].mean(axis=1)
    tmp = pd.DataFrame({"kst_dtm": src["kst_dtm"], "scada_ws": turbine_mean})
    tmp["_hour"] = tmp["kst_dtm"].dt.floor("h")
    tmp = tmp.drop(columns=["kst_dtm"])
    hourly = tmp.groupby("_hour").mean().reset_index()
    return hourly.rename(columns={"_hour": "forecast_kst_dtm"})

def make_group_specific_weight(group, scada_ws):
    weight = np.ones(len(scada_ws))
    if group in ("kpx_group_1", "kpx_group_2"):
        weight = np.where((scada_ws >= 9) & (scada_ws < 12), 5.0, weight)
        weight = np.where(scada_ws >= 12, 3.0, weight)
    else:
        weight = np.where((scada_ws >= 9) & (scada_ws < 12), 2.0, weight)
        weight = np.where((scada_ws >= 12) & (scada_ws < 15), 4.0, weight)
        weight = np.where(scada_ws >= 15, 6.0, weight)
    return weight

wind_correction_models_v4 = {}

for g in TARGET_COLS:
    scada_target = build_scada_wind_target_avg(g)
    df_corr = train_datasets[g][["forecast_kst_dtm"] + RAW_FEATS].merge(
        scada_target, on="forecast_kst_dtm", how="inner"
    )
    df_corr = df_corr.dropna(subset=["scada_ws"])
    tr_corr = df_corr[(df_corr["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df_corr["forecast_kst_dtm"] < CORR_VAL_START)]
    sw = make_group_specific_weight(g, tr_corr["scada_ws"].to_numpy())

    model = LGBMRegressor(
        n_estimators=300, max_depth=4, num_leaves=15, min_child_samples=50,
        learning_rate=0.03, reg_alpha=1.0, reg_lambda=1.0, random_state=42, verbose=-1,
    )
    model.fit(tr_corr[RAW_FEATS], tr_corr["scada_ws"], sample_weight=sw)
    wind_correction_models_v4[g] = model
    print(f"[{g}] 풍속보정모델(v4) 학습 완료")

for g in TARGET_COLS:
    model = wind_correction_models_v4[g]
    train_datasets[g]["corrected_ws_v4"] = model.predict(train_datasets[g][RAW_FEATS])
    test_datasets[g]["corrected_ws_v4"] = model.predict(test_datasets[g][RAW_FEATS])

FEATS_WITH_V4 = RAW_FEATS + ["corrected_ws_v4"]
print(f"\n최종 피처 수 (v4 포함): {len(FEATS_WITH_V4)}")


# ============================================================
# 5. metric 함수 + 모델 팩토리
# ============================================================
def group_score(actual, forecast, capacity):
    valid = actual >= capacity * 0.10
    actual_v, forecast_v = actual[valid], forecast[valid]
    error_rate = np.abs(forecast_v - actual_v) / capacity
    nmae = np.mean(error_rate)
    unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
    earned = np.sum(actual_v * unit_price)
    maxset = np.sum(actual_v * 4.0)
    ficr = earned / maxset
    return 0.5 * (1 - nmae) + 0.5 * ficr, nmae, ficr

def make_model(name):
    if name == "RandomForest":
        return RandomForestRegressor(n_estimators=300, max_depth=16, random_state=42, n_jobs=-1)
    if name == "HistGB":
        return HistGradientBoostingRegressor(max_iter=300, max_depth=None, random_state=42)
    if name == "LightGBM":
        return LGBMRegressor(n_estimators=300, max_depth=-1, num_leaves=63, learning_rate=0.05, random_state=42, verbose=-1)
    if name == "MLP":
        return make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(64, 32), early_stopping=True, random_state=42, max_iter=2000))
    raise ValueError(name)


# ============================================================
# 6. Baseline(일반회귀, top2 중 1등 모델) 학습 - 비교 기준선
# ============================================================
baseline_preds = {}

for g in TARGET_COLS:
    df = train_datasets[g]
    tr = df[(df["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df["forecast_kst_dtm"] < CORR_VAL_START)]
    va = df[df["forecast_kst_dtm"] >= CORR_VAL_START]
    capacity = CAPACITY_KWH[g]

    model = make_model(TOP2_MODELS[g][0])
    model.fit(tr[FEATS_WITH_V4], tr["target"])
    pred = np.clip(model.predict(va[FEATS_WITH_V4]), 0, capacity)
    baseline_preds[g] = pred

    y_va = va["target"].to_numpy(dtype=float)
    score, nmae, ficr = group_score(y_va, pred, capacity)
    print(f"[{g}] baseline({TOP2_MODELS[g][0]}) -> score={score:.4f}")


# ============================================================
# 7. Alpha 세밀 스캔 (quantile regression + seed 앙상블)
# ============================================================
ALPHAS_FINE = [0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.72, 0.75, 0.78, 0.80, 0.85]
SEEDS = [42, 1, 2]

fine_results = []
fine_models_cache = {}

for g in TARGET_COLS:
    df = train_datasets[g]
    tr = df[(df["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df["forecast_kst_dtm"] < CORR_VAL_START)]
    va = df[df["forecast_kst_dtm"] >= CORR_VAL_START].copy()
    capacity = CAPACITY_KWH[g]
    y_va = va["target"].to_numpy(dtype=float)

    print(f"===== {g} =====")
    for alpha in ALPHAS_FINE:
        preds = []
        for seed in SEEDS:
            model = LGBMRegressor(objective='quantile', alpha=alpha, n_estimators=300,
                                   max_depth=8, learning_rate=0.05, random_state=seed, verbose=-1)
            model.fit(tr[FEATS_WITH_V4], tr["target"])
            preds.append(np.clip(model.predict(va[FEATS_WITH_V4]), 0, capacity))

        ensemble_pred = np.mean(preds, axis=0)
        score, nmae, ficr = group_score(y_va, ensemble_pred, capacity)
        print(f"  alpha={alpha:.2f} -> score={score:.4f} nmae={nmae:.4f} ficr={ficr:.4f}")
        fine_results.append({"group": g, "alpha": alpha, "score": score, "nmae": nmae, "ficr": ficr})
        fine_models_cache[(g, alpha)] = ensemble_pred
    print()

fine_df = pd.DataFrame(fine_results)
print("=" * 70)
print(fine_df.pivot(index="alpha", columns="group", values="score").round(4))

best_alpha_per_group = fine_df.loc[fine_df.groupby("group")["score"].idxmax()].set_index("group")["alpha"].to_dict()
print("\n그룹별 최적 alpha:", best_alpha_per_group)


# ============================================================
# 8. 구간별(발전량 bin) 오차 비교 - baseline vs alpha=0.5 vs 최적alpha
# ============================================================
for g in TARGET_COLS:
    va = train_datasets[g][train_datasets[g]["forecast_kst_dtm"] >= CORR_VAL_START].copy()
    capacity = CAPACITY_KWH[g]

    va["pred_baseline"] = baseline_preds[g]
    va["pred_alpha05"] = fine_models_cache[(g, 0.5)]
    va["pred_best"] = fine_models_cache[(g, best_alpha_per_group[g])]

    va["target_pct"] = va["target"] / capacity
    va["target_bin"] = pd.cut(va["target_pct"], bins=[0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.01],
                                labels=["0-10%", "10-30%", "30-50%", "50-70%", "70-90%", "90-100%"])

    for col in ["pred_baseline", "pred_alpha05", "pred_best"]:
        va[f"signed_error_{col}"] = va[col] - va["target"]

    print(f"===== {g} (최적alpha={best_alpha_per_group[g]}) =====")
    summary = va.groupby("target_bin", observed=True).agg(
        n=("target", "size"),
        target_mean=("target", "mean"),
        baseline_err=("signed_error_pred_baseline", "mean"),
        alpha05_err=("signed_error_pred_alpha05", "mean"),
        best_err=("signed_error_pred_best", "mean"),
    )
    print(summary.round(1))
    print()

quantile regression 으로 과대예측해서 전체적인 성능은 좋아지지만, 10% - 70% 구간은 성능이 떨어짐 -> 이 부분을 어떻게 극복할 것인지?

In [ ]:
# 10-70% 구간이 실제 오차율(%)로 얼마나 되는지, 그리고 채점에 얼마나 기여하는지 재확인
for g in TARGET_COLS:
    va = train_datasets[g][train_datasets[g]["forecast_kst_dtm"] >= CORR_VAL_START].copy()
    capacity = CAPACITY_KWH[g]
    va["pred_best"] = fine_models_cache[(g, best_alpha_per_group[g])]
    va["error_rate"] = np.abs(va["pred_best"] - va["target"]) / capacity
    va["target_pct"] = va["target"] / capacity
    va["target_bin"] = pd.cut(va["target_pct"], bins=[0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.01],
                                labels=["0-10%", "10-30%", "30-50%", "50-70%", "70-90%", "90-100%"])
    
    print(f"[{g}]")
    print(va.groupby("target_bin", observed=True)["error_rate"].mean().round(4))
    print()

In [ ]:
# ============================================================
# 최적 alpha 예측 vs 실제값 - 1개월 단위, capa 10% 라인 + 오차율별 색상
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches

for g in TARGET_COLS:
    va = train_datasets[g][train_datasets[g]["forecast_kst_dtm"] >= CORR_VAL_START].copy()
    va = va.sort_values("forecast_kst_dtm").reset_index(drop=True)
    va["pred_best"] = fine_models_cache[(g, best_alpha_per_group[g])]

    capacity = CAPACITY_KWH[g]
    cap10 = capacity * 0.10

    va["error_rate"] = np.abs(va["pred_best"] - va["target"]) / capacity
    # 오차율 구간별 색상 지정
    def err_color(e):
        if e <= 0.06:
            return "tab:green"
        elif e <= 0.08:
            return "tab:orange"
        else:
            return "tab:red"
    va["err_color"] = va["error_rate"].apply(err_color)

    months = [
        ("2024-01-01", "2024-02-01"), ("2024-02-01", "2024-03-01"),
        ("2024-03-01", "2024-04-01"), ("2024-04-01", "2024-05-01"),
        ("2024-05-01", "2024-06-01"), ("2024-06-01", "2024-07-01"),
        ("2024-07-01", "2024-08-01"), ("2024-08-01", "2024-09-01"),
        ("2024-09-01", "2024-10-01"), ("2024-10-01", "2024-11-01"),
        ("2024-11-01", "2024-12-01"), ("2024-12-01", "2025-01-01"),
    ]

    fig, axes = plt.subplots(12, 1, figsize=(16, 40), sharex=False)
    fig.suptitle(f"{g} (alpha={best_alpha_per_group[g]})", fontsize=14)

    legend_handles = [
        mpatches.Patch(color="tab:green", label="error<=6% (price 4.0)"),
        mpatches.Patch(color="tab:orange", label="6%<error<=8% (price 3.0)"),
        mpatches.Patch(color="tab:red", label="error>8% (price 0)"),
        plt.Line2D([0], [0], color="black", linewidth=0.8, label="Actual"),
        plt.Line2D([0], [0], color="gray", linestyle="--", linewidth=1, label="Capacity 10%"),
    ]

    for ax, (start, end) in zip(axes, months):
        sub = va[(va["forecast_kst_dtm"] >= start) & (va["forecast_kst_dtm"] < end)]

        ax.plot(sub["forecast_kst_dtm"], sub["target"], color="black", linewidth=0.8, alpha=0.7, zorder=2)
        ax.scatter(sub["forecast_kst_dtm"], sub["pred_best"], c=sub["err_color"], s=6, zorder=3)
        ax.axhline(cap10, color="gray", linestyle="--", linewidth=1, zorder=1)

        ax.set_title(start[:7])
        ax.set_ylabel("kWh")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d"))
        ax.grid(alpha=0.3)

    axes[0].legend(handles=legend_handles, loc="upper right", fontsize=7, ncol=2)

    plt.tight_layout()
    plt.savefig(f"{g}_actual_vs_pred_monthly_colored.png", dpi=120)
    plt.show()
    print(f"Saved: {g}_actual_vs_pred_monthly_colored.png\n")

# Delta Feature 추가해서 진행
 - 50 min-max는 제외하고 나머지에 대해선 모두 Shift(1) 반영해서 진행
 - group     kpx_group_1  kpx_group_2  kpx_group_3
lag                                            
lag1h          0.6489       0.6617       0.5875
lag2h          0.6485       0.6632       0.5849
lag3h          0.6472       0.6623       0.5863
lag4h          0.6437       0.6611       0.5898
lag5h          0.6494       0.6630       0.5934
no_delta       0.6462       0.6624       0.5888

 - lag 효과는 그렇게 크지가 않은거 같네, 그러면 급격하게 변화하는 거에 대해서 어떻게 대응할지, 풍속예측을 좀 더 디테일하게 하는게 차라리 나은 거 같고, 변수들을 좀 줄일 필요가 있나..?
 
# 20260720 결과
 - 발전량 과대예측하는 경우가 성능이 더 올라감, 하지만 이 과대 예측으로 인해 발전량 90~100%인 구간만 줄어들뿐 나머지 부분은 오히려 Error가 더 커지는 경향이 있음
 - 이를 잡기 위해 플랏을 보고 급격한 변화에 잘 대응하는 모형을 만들기 위해 lag을 도입했으나 효과는 그리 크지 않음
 - 다른 방안이 필요한 것 같고, 현재 변수 수도 너무 많아서 좀 줄일 필요가 있지 않을까 싶음
 - 추가적으로, 현재 격자평균을 사용하고 있는데, 이걸 좀 더 디테일하게 바꿔주고, 풍속 보정 모델을 좀 건드려보면 좋지 않을까 생각

In [12]:
# ============================================================
# 0. 라이브러리 & 경로 설정
# ============================================================
import numpy as np
import pandas as pd
from pathlib import Path
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

DATA_DIR = Path("data")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {"kpx_group_1": 21600, "kpx_group_2": 21600, "kpx_group_3": 21000}
ID_COLS = ["forecast_kst_dtm", "data_available_kst_dtm", "grid_id", "latitude", "longitude"]
CORR_VAL_START = pd.Timestamp("2024-01-01")

TRAIN_PERIOD_START = {
    "kpx_group_1": "2023-01-01",
    "kpx_group_2": "2022-01-01",
    "kpx_group_3": "2023-01-01",
}

GROUP_LDAPS_GRIDS = {
    "kpx_group_1": [5, 6, 10, 11],
    "kpx_group_2": list(range(1, 17)),
    "kpx_group_3": list(range(1, 17)),
}
GFS_GRID = 5

GROUP_SCADA_TURBINES = {
    "kpx_group_1": ("vestas", list(range(1, 7))),
    "kpx_group_2": ("vestas", list(range(7, 13))),
    "kpx_group_3": ("unison", list(range(1, 6))),
}

TOP2_MODELS = {
    "kpx_group_1": ["LightGBM", "HistGB"],
    "kpx_group_2": ["HistGB", "LightGBM"],
    "kpx_group_3": ["MLP", "RandomForest"],
}


# ============================================================
# 1. 원본 파일 로더
# ============================================================
def load_csv(path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    return df

def load_train_labels():
    df = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
    df["kst_dtm"] = pd.to_datetime(df["kst_dtm"])
    return df

def load_scada(maker):
    df = pd.read_csv(TRAIN_DIR / f"scada_{maker}_train.csv", encoding="utf-8-sig")
    df["kst_dtm"] = pd.to_datetime(df["kst_dtm"])
    return df


# ============================================================
# 2. 격자 집계
# ============================================================
def aggregate_grids(df, grids, prefix):
    sub = df[df["grid_id"].isin(grids)]
    value_cols = [c for c in sub.columns if c not in ID_COLS]
    agg = sub.groupby("forecast_kst_dtm")[value_cols].mean()
    agg.columns = [f"{prefix}_{c}" for c in agg.columns]
    return agg.reset_index()


# ============================================================
# 3. 캘린더 변수
# ============================================================
def calendar_features(dt_series):
    dt = pd.to_datetime(dt_series)
    out = pd.DataFrame(index=dt.index)
    out["hour_sin"] = np.sin(2 * np.pi * dt.dt.hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * dt.dt.hour / 24)
    out["month_sin"] = np.sin(2 * np.pi * dt.dt.month / 12)
    out["month_cos"] = np.cos(2 * np.pi * dt.dt.month / 12)
    return out


# ============================================================
# 4. 데이터셋 조립
#    - gust_upper(50m 재정의, 옵션B) 생성
#    - 50m 원본(max/min)은 delta 대상에서 제외
#    - 나머지 모든 U/V 바람 변수(+gust_upper)에 delta1h 적용
#    - delta는 반드시 라벨 병합(dropna) 이전, 시간 연속 상태에서 계산
# ============================================================
def build_dataset(split, group, ldaps, gfs, labels=None):
    ldaps_agg = aggregate_grids(ldaps, GROUP_LDAPS_GRIDS[group], "ldaps")
    gfs_agg = aggregate_grids(gfs, [GFS_GRID], "gfs")

    df = ldaps_agg.merge(gfs_agg, on="forecast_kst_dtm", how="inner")
    df = df.sort_values("forecast_kst_dtm").reset_index(drop=True)

    # ---------- 시간 연속성 확인 ----------
    time_diffs = df["forecast_kst_dtm"].diff().dropna()
    n_gaps = (time_diffs != pd.Timedelta(hours=1)).sum()
    if n_gaps > 0:
        print(f"  [경고][{group}/{split}] 원본 기상데이터에 1시간 간격이 아닌 곳 {n_gaps}개 발견")

    # ---------- 옵션B: 50m gust_upper 재정의 (U/V 비동기 극값 문제 해결한 안전한 버전) ----------
    u_absmax = np.maximum(df["ldaps_heightAboveGround_50_50MUmax"].abs(), df["ldaps_heightAboveGround_50_50MUmin"].abs())
    v_absmax = np.maximum(df["ldaps_heightAboveGround_50_50MVmax"].abs(), df["ldaps_heightAboveGround_50_50MVmin"].abs())
    df["ldaps_ws50_gust_upper"] = np.sqrt(u_absmax**2 + v_absmax**2)

    # ---------- delta 계산 대상 컬럼 (50m 원본 max/min은 제외, gust_upper는 포함) ----------
    delta_target_cols = [c for c in df.columns
                          if c not in ID_COLS
                          and c != "forecast_kst_dtm"
                          and ("_u" in c.lower() or "_v" in c.lower() or "blws" in c.lower() or "gust_upper" in c.lower())
                          and "50_50m" not in c]  # 50m 원본(max/min) 컬럼명에 "50_50M" 패턴 포함되므로 제외

    print(f"  [{group}/{split}] delta 생성 대상 컬럼 수: {len(delta_target_cols)}")
    print(f"    {delta_target_cols}")

    for col in delta_target_cols:
        df[f"{col}_delta1h"] = df[col].diff().fillna(0)

    # ---------- 캘린더 추가 ----------
    df = pd.concat([df, calendar_features(df["forecast_kst_dtm"])], axis=1)

    # ---------- 라벨 병합 (구멍 생겨도 delta는 이미 계산 완료라 안전) ----------
    if labels is not None:
        y = labels[["kst_dtm", group]].rename(columns={"kst_dtm": "forecast_kst_dtm", group: "target"})
        df = df.merge(y, on="forecast_kst_dtm", how="left").dropna(subset=["target"])

    df = df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    feat_cols = [c for c in df.columns if c not in ("forecast_kst_dtm", "target")]
    df[feat_cols] = df[feat_cols].interpolate(method="linear", limit_direction="both")
    return df


# ============================================================
# 5. 실행 - train/test 데이터셋 생성
# ============================================================
ldaps_train_raw = load_csv(TRAIN_DIR / "ldaps_train.csv")
gfs_train_raw = load_csv(TRAIN_DIR / "gfs_train.csv")
ldaps_test_raw = load_csv(TEST_DIR / "ldaps_test.csv")
gfs_test_raw = load_csv(TEST_DIR / "gfs_test.csv")
train_labels = load_train_labels()

train_datasets = {}
test_datasets = {}
for g in TARGET_COLS:
    print(f"=== {g} train ===")
    train_datasets[g] = build_dataset("train", g, ldaps_train_raw, gfs_train_raw, train_labels)
    print(f"=== {g} test ===")
    test_datasets[g] = build_dataset("test", g, ldaps_test_raw, gfs_test_raw, None)

RAW_FEATS = [c for c in train_datasets["kpx_group_1"].columns if c not in ("forecast_kst_dtm", "target")]

for g in TARGET_COLS:
    print(f"[{g}] train={train_datasets[g].shape}, test={test_datasets[g].shape}")
print(f"\n전체 피처 수 (gust_upper + delta 포함): {len(RAW_FEATS)}")
print("delta 컬럼:", [c for c in RAW_FEATS if "delta" in c])


# ============================================================
# 6. SCADA 로드 + 풍속 보정모델(v4) 학습
# ============================================================
scada_vestas = load_scada("vestas")
scada_unison = load_scada("unison")

def build_scada_wind_target_avg(group):
    maker, turbines = GROUP_SCADA_TURBINES[group]
    src = scada_vestas if maker == "vestas" else scada_unison
    ws_cols = [f"{maker}_wtg{t:02d}_ws" for t in turbines]

    turbine_mean = src[ws_cols].mean(axis=1)
    tmp = pd.DataFrame({"kst_dtm": src["kst_dtm"], "scada_ws": turbine_mean})
    tmp["_hour"] = tmp["kst_dtm"].dt.floor("h")
    tmp = tmp.drop(columns=["kst_dtm"])
    hourly = tmp.groupby("_hour").mean().reset_index()
    return hourly.rename(columns={"_hour": "forecast_kst_dtm"})

def make_group_specific_weight(group, scada_ws):
    weight = np.ones(len(scada_ws))
    if group in ("kpx_group_1", "kpx_group_2"):
        weight = np.where((scada_ws >= 9) & (scada_ws < 12), 5.0, weight)
        weight = np.where(scada_ws >= 12, 3.0, weight)
    else:
        weight = np.where((scada_ws >= 9) & (scada_ws < 12), 2.0, weight)
        weight = np.where((scada_ws >= 12) & (scada_ws < 15), 4.0, weight)
        weight = np.where(scada_ws >= 15, 6.0, weight)
    return weight

wind_correction_models_v4 = {}

for g in TARGET_COLS:
    scada_target = build_scada_wind_target_avg(g)
    df_corr = train_datasets[g][["forecast_kst_dtm"] + RAW_FEATS].merge(
        scada_target, on="forecast_kst_dtm", how="inner"
    )
    df_corr = df_corr.dropna(subset=["scada_ws"])
    tr_corr = df_corr[(df_corr["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df_corr["forecast_kst_dtm"] < CORR_VAL_START)]
    sw = make_group_specific_weight(g, tr_corr["scada_ws"].to_numpy())

    model = LGBMRegressor(
        n_estimators=300, max_depth=4, num_leaves=15, min_child_samples=50,
        learning_rate=0.03, reg_alpha=1.0, reg_lambda=1.0, random_state=42, verbose=-1,
    )
    model.fit(tr_corr[RAW_FEATS], tr_corr["scada_ws"], sample_weight=sw)
    wind_correction_models_v4[g] = model
    print(f"[{g}] 풍속보정모델(v4) 학습 완료")

for g in TARGET_COLS:
    model = wind_correction_models_v4[g]
    train_datasets[g]["corrected_ws_v4"] = model.predict(train_datasets[g][RAW_FEATS])
    test_datasets[g]["corrected_ws_v4"] = model.predict(test_datasets[g][RAW_FEATS])

FEATS_WITH_V4 = RAW_FEATS + ["corrected_ws_v4"]
print(f"\n최종 피처 수 (gust_upper + delta + v4): {len(FEATS_WITH_V4)}")


# ============================================================
# 7. metric 함수 + 모델 팩토리
# ============================================================
def group_score(actual, forecast, capacity):
    valid = actual >= capacity * 0.10
    actual_v, forecast_v = actual[valid], forecast[valid]
    error_rate = np.abs(forecast_v - actual_v) / capacity
    nmae = np.mean(error_rate)
    unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
    earned = np.sum(actual_v * unit_price)
    maxset = np.sum(actual_v * 4.0)
    ficr = earned / maxset
    return 0.5 * (1 - nmae) + 0.5 * ficr, nmae, ficr

def make_model(name):
    if name == "RandomForest":
        return RandomForestRegressor(n_estimators=300, max_depth=16, random_state=42, n_jobs=-1)
    if name == "HistGB":
        return HistGradientBoostingRegressor(max_iter=300, max_depth=None, random_state=42)
    if name == "LightGBM":
        return LGBMRegressor(n_estimators=300, max_depth=-1, num_leaves=63, learning_rate=0.05, random_state=42, verbose=-1)
    if name == "MLP":
        return make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(64, 32), early_stopping=True, random_state=42, max_iter=2000))
    raise ValueError(name)


# ============================================================
# 8. Baseline(일반회귀, delta+gust_upper 포함) 학습
# ============================================================
baseline_preds = {}

for g in TARGET_COLS:
    df = train_datasets[g]
    tr = df[(df["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df["forecast_kst_dtm"] < CORR_VAL_START)]
    va = df[df["forecast_kst_dtm"] >= CORR_VAL_START]
    capacity = CAPACITY_KWH[g]

    model = make_model(TOP2_MODELS[g][0])
    model.fit(tr[FEATS_WITH_V4], tr["target"])
    pred = np.clip(model.predict(va[FEATS_WITH_V4]), 0, capacity)
    baseline_preds[g] = pred

    y_va = va["target"].to_numpy(dtype=float)
    score, nmae, ficr = group_score(y_va, pred, capacity)
    print(f"[{g}] baseline+gust_upper+delta({TOP2_MODELS[g][0]}) -> score={score:.4f}")

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [ ]:
# ============================================================
# 8. Quantile Regression (alpha 세밀스캔) + Seed 앙상블
#    - gust_upper + delta 피처 포함된 FEATS_WITH_V4로 학습
# ============================================================
ALPHAS_FINE = [0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.72, 0.75, 0.78, 0.80, 0.85]
SEEDS = [42, 1, 2]

fine_results = []
fine_models_cache = {}

for g in TARGET_COLS:
    df = train_datasets[g]
    tr = df[(df["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df["forecast_kst_dtm"] < CORR_VAL_START)]
    va = df[df["forecast_kst_dtm"] >= CORR_VAL_START].copy()
    capacity = CAPACITY_KWH[g]
    y_va = va["target"].to_numpy(dtype=float)

    print(f"===== {g} =====")
    for alpha in ALPHAS_FINE:
        preds = []
        for seed in SEEDS:
            model = LGBMRegressor(objective='quantile', alpha=alpha, n_estimators=300,
                                   max_depth=8, learning_rate=0.05, random_state=seed, verbose=-1)
            model.fit(tr[FEATS_WITH_V4], tr["target"])
            preds.append(np.clip(model.predict(va[FEATS_WITH_V4]), 0, capacity))

        ensemble_pred = np.mean(preds, axis=0)
        score, nmae, ficr = group_score(y_va, ensemble_pred, capacity)
        print(f"  alpha={alpha:.2f} -> score={score:.4f} nmae={nmae:.4f} ficr={ficr:.4f}")
        fine_results.append({"group": g, "alpha": alpha, "score": score, "nmae": nmae, "ficr": ficr})
        fine_models_cache[(g, alpha)] = ensemble_pred
    print()

fine_df = pd.DataFrame(fine_results)
print("=" * 70)
print("Quantile alpha별 결과 (delta+gust_upper 포함)")
print("=" * 70)
print(fine_df.pivot(index="alpha", columns="group", values="score").round(4))

best_alpha_per_group = fine_df.loc[fine_df.groupby("group")["score"].idxmax()].set_index("group")["alpha"].to_dict()
print("\n그룹별 최적 alpha:", best_alpha_per_group)


# ============================================================
# 9. 이전(delta 없는) 결과와 비교 - 직접 대조
# ============================================================
# 이전에 확인했던 delta 없는 quantile 최적 결과 (참고용 하드코딩)
PREVIOUS_BEST_NO_DELTA = {
    "kpx_group_1": 0.6466,  # alpha=0.75
    "kpx_group_2": 0.6646,  # alpha=0.60
    "kpx_group_3": 0.5863,  # alpha=0.75
}

print("=" * 70)
print("delta 피처 추가 효과 비교 (이전 vs 지금)")
print("=" * 70)
for g in TARGET_COLS:
    new_best = fine_df[fine_df["group"] == g]["score"].max()
    old_best = PREVIOUS_BEST_NO_DELTA[g]
    print(f"[{g}] delta없음={old_best:.4f} -> delta있음={new_best:.4f}  (차이={new_best-old_best:+.4f})")

In [ ]:
# ============================================================
# 4. 데이터셋 조립 - lag을 파라미터로 받아서, 하나씩 따로 생성 가능하게
# ============================================================
def build_dataset(split, group, ldaps, gfs, labels=None, lag_hours=None):
    """lag_hours: None이면 delta 피처 생성 안 함, 정수(1~5)면 그 lag만 생성"""
    ldaps_agg = aggregate_grids(ldaps, GROUP_LDAPS_GRIDS[group], "ldaps")
    gfs_agg = aggregate_grids(gfs, [GFS_GRID], "gfs")

    df = ldaps_agg.merge(gfs_agg, on="forecast_kst_dtm", how="inner")
    df = df.sort_values("forecast_kst_dtm").reset_index(drop=True)

    time_diffs = df["forecast_kst_dtm"].diff().dropna()
    n_gaps = (time_diffs != pd.Timedelta(hours=1)).sum()
    if n_gaps > 0:
        print(f"  [경고][{group}/{split}] 1시간 간격 아닌 곳 {n_gaps}개")

    u_absmax = np.maximum(df["ldaps_heightAboveGround_50_50MUmax"].abs(), df["ldaps_heightAboveGround_50_50MUmin"].abs())
    v_absmax = np.maximum(df["ldaps_heightAboveGround_50_50MVmax"].abs(), df["ldaps_heightAboveGround_50_50MVmin"].abs())
    df["ldaps_ws50_gust_upper"] = np.sqrt(u_absmax**2 + v_absmax**2)

    if lag_hours is not None:
        delta_target_cols = [c for c in df.columns
                              if c not in ID_COLS
                              and c != "forecast_kst_dtm"
                              and ("_u" in c.lower() or "_v" in c.lower() or "blws" in c.lower() or "gust_upper" in c.lower())
                              and "50_50m" not in c]
        for col in delta_target_cols:
            df[f"{col}_delta{lag_hours}h"] = df[col].diff(lag_hours).fillna(0)

    df = pd.concat([df, calendar_features(df["forecast_kst_dtm"])], axis=1)

    if labels is not None:
        y = labels[["kst_dtm", group]].rename(columns={"kst_dtm": "forecast_kst_dtm", group: "target"})
        df = df.merge(y, on="forecast_kst_dtm", how="left").dropna(subset=["target"])

    df = df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    feat_cols = [c for c in df.columns if c not in ("forecast_kst_dtm", "target")]
    df[feat_cols] = df[feat_cols].interpolate(method="linear", limit_direction="both")
    return df


# ============================================================
# 5. lag별로 각각 데이터셋 생성 + baseline(delta없음) 포함해서 quantile 비교
# ============================================================
ldaps_train_raw = load_csv(TRAIN_DIR / "ldaps_train.csv")
gfs_train_raw = load_csv(TRAIN_DIR / "gfs_train.csv")
train_labels = load_train_labels()
scada_vestas = load_scada("vestas")
scada_unison = load_scada("unison")

LAG_CANDIDATES = [None, 1, 2, 3, 4, 5]  # None = delta 없는 baseline

lag_comparison_results = []

for lag in LAG_CANDIDATES:
    lag_label = "no_delta" if lag is None else f"lag{lag}h"
    print(f"\n{'='*20} {lag_label} {'='*20}")

    train_datasets_lag = {}
    for g in TARGET_COLS:
        train_datasets_lag[g] = build_dataset("train", g, ldaps_train_raw, gfs_train_raw, train_labels, lag_hours=lag)

    for g in TARGET_COLS:
        RAW_FEATS_LAG = [c for c in train_datasets_lag[g].columns if c not in ("forecast_kst_dtm", "target")]

        # 보정모델(v4) 재학습 - 이번 피처셋 기준
        scada_target = build_scada_wind_target_avg(g)
        df_corr = train_datasets_lag[g][["forecast_kst_dtm"] + RAW_FEATS_LAG].merge(scada_target, on="forecast_kst_dtm", how="inner")
        df_corr = df_corr.dropna(subset=["scada_ws"])
        tr_corr = df_corr[(df_corr["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df_corr["forecast_kst_dtm"] < CORR_VAL_START)]
        sw = make_group_specific_weight(g, tr_corr["scada_ws"].to_numpy())

        corr_model = LGBMRegressor(n_estimators=300, max_depth=4, num_leaves=15, min_child_samples=50,
                                     learning_rate=0.03, reg_alpha=1.0, reg_lambda=1.0, random_state=42, verbose=-1)
        corr_model.fit(tr_corr[RAW_FEATS_LAG], tr_corr["scada_ws"], sample_weight=sw)
        train_datasets_lag[g]["corrected_ws_v4"] = corr_model.predict(train_datasets_lag[g][RAW_FEATS_LAG])

        FEATS_LAG = RAW_FEATS_LAG + ["corrected_ws_v4"]

        df = train_datasets_lag[g]
        tr = df[(df["forecast_kst_dtm"] >= TRAIN_PERIOD_START[g]) & (df["forecast_kst_dtm"] < CORR_VAL_START)]
        va = df[df["forecast_kst_dtm"] >= CORR_VAL_START].copy()
        capacity = CAPACITY_KWH[g]
        y_va = va["target"].to_numpy(dtype=float)

        # quantile alpha 대표값 몇 개만 빠르게 비교 (세밀스캔은 나중에)
        best_score = -1
        best_alpha_here = None
        for alpha in [0.5, 0.6, 0.65, 0.7, 0.75]:
            preds = []
            for seed in [42, 1, 2]:
                model = LGBMRegressor(objective='quantile', alpha=alpha, n_estimators=300,
                                       max_depth=8, learning_rate=0.05, random_state=seed, verbose=-1)
                model.fit(tr[FEATS_LAG], tr["target"])
                preds.append(np.clip(model.predict(va[FEATS_LAG]), 0, capacity))
            ensemble_pred = np.mean(preds, axis=0)
            score, nmae, ficr = group_score(y_va, ensemble_pred, capacity)
            if score > best_score:
                best_score = score
                best_alpha_here = alpha

        print(f"  [{g}] n_feats={len(FEATS_LAG):3d}  best_alpha={best_alpha_here}  best_score={best_score:.4f}")
        lag_comparison_results.append({"lag": lag_label, "group": g, "n_feats": len(FEATS_LAG),
                                         "best_alpha": best_alpha_here, "best_score": best_score})

lag_comp_df = pd.DataFrame(lag_comparison_results)
print("\n" + "=" * 70)
print("Lag별 비교 요약")
print("=" * 70)
print(lag_comp_df.pivot(index="lag", columns="group", values="best_score").round(4))